
# PART III: NLP TASK - SENTIMENT ANALYSIS PIPELINE
Student Name: Prachi Rajbanshi | Student ID: 2408861

Module: 6CS012 AI & ML


In [ ]:
!pip uninstall jax jaxlib

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Would remove:
    /usr/local/lib/python3.12/dist-packages/jax-0.7.2.dist-info/*
    /usr/local/lib/python3.12/dist-packages/jax/*
Proceed (Y/n)? 

In [ ]:
!pip install gensim nltk gradio matplotlib seaborn

In [ ]:
# Importing necessary libraries
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import gensim.downloader as api
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Setup and resource downloading (NLTK resources)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

**1. Text Cleaning Pipeline**

In [ ]:
# Text cleaning pipeline
def text_cleaning_pipeline(text):
  # For LOWERCASE
  text = str(text).lower()

  # For REMOVING URLs
  text = re.sub(r"http\S+|www\.\S+", "", text)

  # For REMOVING MENTIONS
  text = re.sub(r"@\w+", "", text)

  # For REMOVING SPECIAL CHARACTERS & PUNCTUATIONS
  text = re.sub(r"[^a-zA-Z0-9\s]", "", text)

  # For TOKENIZING & PROCESSING
  tokens = text.split()

  # For REMOVING STOP-WORDS & APPLY LEMMATIZATION
  tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

  return " ".join(tokens)

**2. Load Dataset**

In [ ]:
# Load the dataset
try:
  data = pd.read_csv("/content/drive/MyDrive/AI&ML-Assessment_PR/2. Hotel Review Dataset/Hotel_Reviews.csv")
  print("Dataset Loaded Successfully\n")
  print("Dataset Used: Hotel Review Dataset\n")
  print("Dataset Shape: ", data.shape, "\n")
except FileNotFoundError:
  print("File not found. Please check the file path.")

data.head()
print()
print("Data Columns:\n",data.columns)


In [ ]:
data.head()

**3. Data Cleaning and Preprocessing**

In [ ]:
# Clean Labels (Converting 1-5 Ratings to Binary [1 for Positive, 0 for Negative])
data['Sentiment'] = data['Rating'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
# Remove any null values to prevent errors during vectorization
data = data.dropna()

# Apply cleaning
data['Clean_review'] = data['Review'].apply(text_cleaning_pipeline)

data[['Review', 'Clean_review']].head()

In [ ]:
# Ensure the Sentiment conversion happens first
# Logic: 4-5 are Positive (1), 1-3 are Negative (0)
# data['Sentiment'] = data['Rating'].apply(lambda x: 1 if x >= 4 else 0)

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Original 1-5 Rating Distribution
# Added data=df to fix the ValueError
sns.countplot(x='Rating', data=data, palette='viridis', ax=axes[0])
axes[0].set_title('Step 1: Original Rating Distribution (1-5)')
axes[0].set_xlabel('Hotel Rating')
axes[0].set_ylabel('Number of Reviews')

# Subplot 2: Processed Binary Sentiment Distribution
# This shows how the model "sees" the data after conversion
sns.countplot(x='Sentiment', data=data, palette='coolwarm', ax=axes[1])
axes[1].set_title('Step 2: Binary Sentiment Distribution (0=Neg, 1=Pos)')
axes[1].set_xlabel('Sentiment Category')
axes[1].set_xticklabels(['Negative (0-3 Stars)', 'Positive (4-5 Stars)'])
axes[1].set_ylabel('Number of Reviews')

plt.tight_layout()
plt.show()

# Print statistics for your report
val_counts = data['Sentiment'].value_counts(normalize=True) * 100
print(f"Class Distribution:\n{data['Sentiment'].value_counts()}")
print(f"\nPercentage Breakdown:\nPositive: {val_counts[1]:.2f}%\nNegative: {val_counts[0]:.2f}%")

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

all_words = ' '.join(data['Clean_review'])

wordcloud = WordCloud(
    width=300,
    height=100,
    background_color='white',
    max_words=100
).generate(all_words)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Top 100 Most Frequent Words')
plt.show()

**4. Train-Test Split**

In [ ]:
# Train-Test Split (80%)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    data['Clean_review'], data['Sentiment'], test_size=0.2, random_state=42
)

**5. Tokenization**

In [ ]:
# Tokenization & Percentile-based padding
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train_text)
vocab_size = len(tokenizer.word_index) + 1

train_sequences = tokenizer.texts_to_sequences(X_train_text)
test_sequences = tokenizer.texts_to_sequences(X_test_text)

**6. Percentile-based Padding (to avoid excessively long sequences)**

In [ ]:
# Calculate 95th percentile length to avoid outliers
max_length = int(np.percentile([len(x) for x in train_sequences], 95))

X_train = pad_sequences(train_sequences, maxlen=max_length, padding='pre')
X_test = pad_sequences(test_sequences, maxlen=max_length, padding='pre')

**7. Model Building**

In [ ]:
# Model 1: Simple RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model1 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length, mask_zero=True),
    SimpleRNN(64, return_sequences=True),
    Dropout(0.3),
    SimpleRNN(32),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Model 2: LSTM (Trainable Embedding)
from tensorflow.keras.optimizers import Adam
model2 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length, mask_zero=True),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model2.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
# Model 3: LSTM with Pre-tained Word2Vec (GloVe)
print("Loading Word2Vec/GloVe Vectors")
w2v_model = api.load('glove-wiki-gigaword-100')

# Prepare Embedding Matrix
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:          # ← add this guard
        continue
    if word in w2v_model:
        embedding_matrix[i] = w2v_model[word]

model3 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=100,
              weights=[embedding_matrix], input_length=max_length,
              trainable=False,
              mask_zero = True), # fine-tuning embeddings to adapt to hotel reviews
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

**8. Model Training**

In [ ]:
# Convert to NumPy explicitly
y_train_np = np.array(y_train).astype('float32')
y_test_np = np.array(y_test).astype('float32')

# Ensure X_train is also a NumPy array (pad_sequences should already do this,
# but it's good to be safe)
X_train_np = np.array(X_train)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

weights = compute_class_weight('balanced', classes=np.array([0,1]), y=y_train)
cw = {0: 1.5, 1: 1.0} # Give Negative 50% more importance, not 200%
cw_lstm = {0: 2.0, 1: 1.0} #stronger needed for LSTM

print("Training Model 1...")
history1 = model1.fit(X_train, y_train, epochs=10, validation_split=0.1, callbacks=[early_stop], class_weight=cw, verbose=0)

print("Training Model 2...")
history2 = model2.fit(X_train, y_train, epochs=10, validation_split=0.1, callbacks=[early_stop], class_weight=cw_lstm, verbose=0)

print("Training Model 3...")
history3 = model3.fit(X_train, y_train, epochs=10, validation_split=0.1, callbacks=[early_stop], class_weight=cw_lstm,verbose=0)

**9. Visualization & Metrices**

In [ ]:
def plot_and_evaluate(model, history_object, X_test, y_test, name): # Added history_object
    # Plot Accuracy
    plt.figure(figsize=(10, 4))
    plt.plot(history_object.history['accuracy'], label='Train')
    plt.plot(history_object.history['val_accuracy'], label='Val')
    plt.title(f'{name} Accuracy Curves')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

    # Evaluation
    preds = (model.predict(X_test) > 0.5).astype(int)
    print(f"\nClassification Report for {name}:")
    print(classification_report(y_test, preds))

    # Confusion Matrix
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix: {name}')
    plt.show()

# Call them like this:
plot_and_evaluate(model1, history1, X_test, y_test, "Model 1: Simple RNN")
plot_and_evaluate(model2, history2, X_test, y_test, "Model 2: LSTM")
plot_and_evaluate(model3, history3, X_test, y_test, "Model 3: Word2Vec + LSTM")

In [ ]:
def plot_history(histories, model_names):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot Accuracy
    for i, history in enumerate(histories):
        axes[0].plot(history.history['accuracy'], label=f'{model_names[i]} Train')
        axes[0].plot(history.history['val_accuracy'], linestyle='--', label=f'{model_names[i]} Val')
    axes[0].set_title('Model Accuracy Comparison')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    # Plot Loss
    for i, history in enumerate(histories):
        axes[1].plot(history.history['loss'], label=f'{model_names[i]} Train')
        axes[1].plot(history.history['val_loss'], linestyle='--', label=f'{model_names[i]} Val')
    axes[1].set_title('Model Loss Comparison')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.show()

# Call the function
plot_history([history1, history2, history3], ['Simple RNN', 'LSTM', 'Word2Vec+LSTM'])

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import seaborn as sns

def evaluate_model(model, X_test, y_test, name):
    # Get predictions
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int)

    print(f"\n{'='*20} {name} Evaluation {'='*20}")

    # Accuracy Score
    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix Visualization
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

# Evaluate each model
evaluate_model(model1, X_test, y_test, "Simple RNN")
evaluate_model(model2, X_test, y_test, "LSTM")
evaluate_model(model3, X_test, y_test, "Word2Vec + LSTM")

In [ ]:
# Error analysis — show misclassified examples
y_pred = (model2.predict(X_test) > 0.5).astype(int).flatten()
errors_idx = np.where(y_pred != y_test.values)[0]

print(f"Total misclassified: {len(errors_idx)}")
for i in errors_idx[:3]:
    print(f"\nReview   : {X_test_text.values[i][:120]}")
    print(f"True     : {'Positive' if y_test.values[i]==1 else 'Negative'}")
    print(f"Predicted: {'Positive' if y_pred[i]==1 else 'Negative'}")

In [ ]:
import gradio as gr
import numpy as np

def live_prediction(text, model_choice):
    # 1. Preprocess the input text
    cleaned = text_cleaning_pipeline(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_length)

    # 2. Select the model based on user dropdown choice
    if model_choice == "Simple RNN":
        selected_model = model1
    elif model_choice == "LSTM":
        selected_model = model2
    else:
        selected_model = model3 # LSTM + Word2Vec

    # 3. Predict
    prediction = selected_model.predict(padded)[0][0]

    # 4. Format Output
    label = "Positive Sentiment" if prediction >= 0.5 else "Negative Sentiment"
    confidence = float(prediction if prediction >= 0.5 else 1 - prediction)

    return {
        "Label": label,
        "Confidence": f"{round(confidence * 100, 2)}%",
        "Cleaned Text": cleaned,
        "Raw Probability": f"{round(float(prediction), 4)}"
    }

# Build the Gradio Interface
interface = gr.Interface(
    fn=live_prediction,
    inputs=[
        gr.Textbox(lines=3, placeholder="Enter a hotel review here...", label="Input Review"),
        gr.Dropdown(["Simple RNN", "LSTM", "LSTM + Word2Vec"], label="Select Model", value="LSTM + Word2Vec")
    ],
    outputs=gr.JSON(label="Analysis Results"),
    title="Hotel Review Sentiment Analyzer",
    description="Compare our three Deep Learning models (RNN, LSTM, and Word2Vec) on real-time hotel feedback.",
    examples=[
        ["The room was incredibly clean and the staff were so helpful!", "LSTM + Word2Vec"],
        ["The bathroom was disgusting and there was loud noise all night.", "LSTM"],
        ["It was an okay stay, but the breakfast was cold.", "Simple RNN"]
    ]
)

# Launch with share=True for the public link needed for your report
interface.launch(share=True)

In [ ]:
import gradio as gr

def live_prediction(text):
    cleaned = text_cleaning_pipeline(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=max_length)
    score = model3.predict(padded)[0][0]
    return "Positive Sentiment" if score > 0.7 else "Negative Sentiment"

demo = gr.Interface(fn=live_prediction, inputs="text", outputs="text", title="Hotel Review Sentiment Analyzer")
demo.launch()